# LangGraph One-Investigator Smoke Test — Restricted MVP Experiment

## 1 — Objective and workflow diagram

A deliberately **restricted** validation of the multi-agent orchestration
flow — the safest possible first step before any paid model experiment:

1. the Planner produces a plan (fixture, one step: **Plan 1**);
2. the Query Router / scheduler selects exactly that one task;
3. the LangGraph orchestrator executes **only** that task;
4. one Investigator processes Plan 1 and returns evidence with provenance;
5. the workflow **terminates immediately** after that result.

**Mandatory defaults:** zero external model-provider calls, zero retries,
no automatic replanning, no second Investigator, Analyst disabled.
Every limit below is enforced in executable code — never by a prompt.

```mermaid
flowchart LR
    Q["question"] --> P["Planner fixture<br/>(offline double)"]
    P --> SEL["selection: ready_tasks<br/>exactly Plan 1 (P1)"]
    SEL --> R["QueryRouter<br/>P1 to architecture specialist"]
    R --> G["M21 LangGraph graph<br/>1 Investigator task, retries=0"]
    G --> C["commit result + evidence"]
    C --> T["terminate: NO_ANALYSIS<br/>no Analyst, no 2nd task"]
    POL["OneTaskExecutionPolicy<br/>max 1 execute (test-only)"] -.blocks.-> G
    GUARD["ProviderCallGuard<br/>limit = 0"] -.blocks.-> NET[("model provider")]
```

Sections 2–9 are the DRY run (the executed default). Section 10 is the
opt-in live experiment (disabled). Section 11 records results,
limitations, and the next step.

## 2 — Environment and dependency checks

Sets the two mandatory switches to their safe defaults
(`LIVE_MODEL_TEST = False`, `CONFIRM_LIVE_CALLS = False`), loads the
**test-only smoke policy** by path (same pattern the M21 notebook uses
for `m17_doubles`), and arms the **provider guard** on the single
provider adapter (`OpenAILLMClient.complete`) with `limit = 0` — from
this point, any real model call raises instead of spending money.

In [ ]:
import importlib.util
import platform
import tempfile
from pathlib import Path


def find_repo_root():
    """Repo location without machine-specific paths: walk up from the
    working directory, fall back to the editable install layout."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "repo_intelligence"
        ).is_dir():
            return candidate
    import repo_intelligence

    return Path(repo_intelligence.__file__).resolve().parents[2]


ROOT = find_repo_root()
WORK = Path(tempfile.mkdtemp(prefix="langgraph-one-investigator-"))

# --- mandatory defaults: dry run only -------------------------------------
LIVE_MODEL_TEST = False        # section 10 stays off unless flipped ON
CONFIRM_LIVE_CALLS = False     # separate, deliberate confirmation step
LIVE_MAX_OUTPUT_TOKENS = 2048  # provider output cap for the live call
LIVE_MAX_INPUT_TOKENS = 8000   # ESTIMATED input ceiling (see section 10)
LIVE_MAX_PROVIDER_CALLS = 1    # hard cap: ONE paid call, enforced below

# --- optional graph dependency --------------------------------------------
try:
    import langgraph  # noqa: F401
    from importlib import metadata

    GRAPH_AVAILABLE = True
    GRAPH_REASON = "langgraph " + metadata.version("langgraph")
except ImportError as exc:
    GRAPH_AVAILABLE = False
    GRAPH_REASON = "missing optional graph extra ({})".format(exc)
assert GRAPH_AVAILABLE, GRAPH_REASON

# --- test-only smoke policy (path import, like m17_doubles) ---------------
POLICY_PATH = ROOT / "tests" / "agents" / "one_investigator_smoke_policy.py"
_spec = importlib.util.spec_from_file_location(
    "one_investigator_smoke_policy", POLICY_PATH
)
smoke_policy = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(smoke_policy)

# --- ZERO-provider-call enforcement (armed BEFORE anything else) ----------
PROVIDER_GUARD = smoke_policy.install_provider_guard(limit=0)
assert PROVIDER_GUARD.limit == 0 and PROVIDER_GUARD.calls == 0

print("python        :", platform.python_version())
print("repo root     :", ROOT.name)
print("langgraph     :", GRAPH_REASON)
print("provider guard: ARMED, limit = 0 (any real model call raises)")
print("LIVE_MODEL_TEST    :", LIVE_MODEL_TEST)
print("CONFIRM_LIVE_CALLS :", CONFIRM_LIVE_CALLS)
print("live caps (calls/output tokens):",
      LIVE_MAX_PROVIDER_CALLS, "/", LIVE_MAX_OUTPUT_TOKENS)
print("live input ceiling : ~{} ESTIMATED input tokens"
      .format(LIVE_MAX_INPUT_TOKENS))
print("workdir       :", WORK.name)

## 3 — Existing component imports and contracts

Every participant and seam below is an existing repository component —
no second Planner, Router, Investigator framework, or orchestration
engine is introduced anywhere in this notebook.

In [ ]:
from repo_intelligence import repository_query as rq
from repo_intelligence.agent import RequestBudgetExceededError
from repo_intelligence.llm.models import LLMRequest
from repo_intelligence.orchestration import (
    INCOMPLETE,
    TASK_COMPLETED,
    TASK_FAILED,
    AgentTaskExecutor,
    OrchestrationConfig,
    Orchestrator,
    assign_roles,
    ready_tasks,
    sequential_dependencies,
)
from repo_intelligence.orchestration.models import TaskRecord
from repo_intelligence.multiagent import TaskPayload

print("task-loop seam :", rq._run_with_langgraph.__name__)
print("planner seam   :", rq._build_plan.__name__,
      "(H1 request-budget preflight inside)")
print("planner contract:", "build_planning_request -> enforce -> "
      "complete -> plan_from_response -> validate_plan")
print("config defaults: max_retries =", OrchestrationConfig().max_retries,
      "| smoke override in section 4: 0")
print("executor contract: AgentExecutor.execute(request) -> result")
print("request fields :", list(LLMRequest.__dataclass_fields__))

## 4 — Deterministic, no-provider-call dry run

The composition reuses the REAL production pieces (offline scripted
models, `QueryRouter`, `AgentTaskExecutor`, the M21 graph seam) and
wraps them in executable restrictions:

| Restriction | Mechanism (code, not prompt) |
|---|---|
| ≤ 1 executed task, only Plan 1 | `OneTaskExecutionPolicy` — test-only wrapper over the existing `AgentExecutor` contract |
| 0 retries | `OrchestrationConfig(max_retries=0)` |
| Analyst never invoked | `AnalystForbidden` participant double (records, then raises) |
| 0 provider calls | `ProviderCallGuard(limit=0)` — any `OpenAILLMClient.complete` raises |
| 1-step plan | fixture `one_task_plan_payload` (defense-in-depth; the policy is the enforcement) |

**Documented limitation:** the M21 graph has no built-in “stop after N
tasks” switch, and a broad graph refactor is out of scope for this
smoke test — so the max-1 rule is enforced by the small test-only
execution policy above, at the existing task-execution boundary.

In [ ]:
# --- the dry-run composition (existing components only) -------------------
QUESTION = "How do the agents, tools, and MCP servers interact?"
LABEL = "one-investigator-smoke"
COMMIT = rq.SUBJECT_COMMIT

models = rq.build_offline_models(QUESTION, LABEL, COMMIT, "README.md")
planner = rq.OfflineModel(
    rq.structured_response(smoke_policy.one_task_plan_payload(QUESTION))
)
sink = rq.JsonFileTraceSink(WORK / "traces")
manager = rq.build_memory_manager(str(WORK / "memory.db"))
memory_context = rq.build_memory_context(LABEL, COMMIT)
engine = rq.build_engine(LABEL)
bindings = rq.fixture_bindings(rq.default_subject_repository())

architect = rq.build_architect(
    models["arch"], sink, manager, memory_context, engine,
    bindings=bindings,
)
flow_investigator = rq.build_flow_investigator(
    models["flow"], sink, manager, memory_context, engine,
    bindings=bindings,
)
router = rq.QueryRouter(architect, flow_investigator)
analyst = smoke_policy.AnalystForbidden()

SMOKE_CONFIG = OrchestrationConfig(max_retries=0)   # 0 retries, in code
policy = smoke_policy.OneTaskExecutionPolicy(
    AgentTaskExecutor.for_agents(router, analyst),
    allowed_task_id="P1",
)
orchestrator = Orchestrator(
    router, analyst,
    trace_sink=sink,
    config=SMOKE_CONFIG,
    executor=policy,
)
task = TaskPayload(
    question=QUESTION, repository_scope=LABEL, commit_scope=COMMIT
)

print("composition ready")
print("  retries allowed :", SMOKE_CONFIG.max_retries)
print("  policy allows   :", policy._allowed_task_id,
      "| max executed:", policy._max_executed)
print("  provider guard  : limit", PROVIDER_GUARD.limit)
print("  participants    : planner=offline, investigator=offline,",
      "analyst=forbidden")

## 5 — Planner output inspection

The plan is built through the SAME production seam the CLI uses:
`_build_plan` measures the complete request against the H1 request
budget **before** any client is asked to answer, then parses and
validates the fixture plan. The planner double is offline — one local
turn, zero provider calls.

In [ ]:
plan = rq._build_plan(planner, QUESTION)

print("objective :", plan.objective)
print("steps ({}) :".format(len(plan.steps)))
for step in plan.steps:
    print("   ", step.id, "|", step.status, "|", step.goal)
assert [step.id for step in plan.steps] == ["P1"], "one-step fixture"
assert len(planner.requests) == 1, "offline planner answered once"

roles = assign_roles(plan.steps)
print("roles     :", [(r.step_id, r.role) for r in roles])
print("planner   :", planner.model,
      "| provider calls so far:", PROVIDER_GUARD.calls)

## 6 — Query Router selection of Plan 1

Two existing, pure views of “which task runs next” — inspected WITHOUT
executing anything: the scheduler's `ready_tasks` (dependency
readiness) selects Plan 1, and the `QueryRouter`'s existing precedence
classifies P1's goal to the architecture specialist. Roles are shown as
the production seam preassigns them.

In [ ]:
# scheduler view: Plan 1 is the only ready task
dependencies = sequential_dependencies([step.id for step in plan.steps])
preview_records = [
    TaskRecord(
        task_id=step.id, role="INVESTIGATOR",
        depends_on=dependencies[step.id],
    )
    for step in plan.steps
]
selected_ids = [record.task_id for record in ready_tasks(preview_records)]
print("ready tasks     :", selected_ids, "-> exactly Plan 1")
assert selected_ids == ["P1"]

# router view: existing precedence, inspected without running a model
route_label = rq.QueryRouter._by_goal(plan.steps[0].goal)
print("goal route      :", route_label, "(existing keyword families)")
assert route_label == "architecture"

print("preassigned role:", [(r.step_id, r.role) for r in roles])
assert [(r.step_id, r.role) for r in roles] == [("P1", "INVESTIGATOR")]

## 7 — LangGraph execution of exactly one Investigator task

ONE graph execution through the production seam
(`_run_with_langgraph`): roles are preassigned with the same
`assign_roles` the legacy executor uses, then the M21 graph owns
select → execute → commit → terminate. The one-task policy wraps the
executor, so a second dispatch would fail loudly before reaching any
participant.

In [ ]:
run = rq._run_with_langgraph(orchestrator, task, plan)

print("status     :", run.status, "| termination:", run.termination_reason)
print("retries    :", run.retries, "| completed:", run.completed_steps)
print("tasks      :", [
    (t.task_id, t.role, t.status, "attempts={}".format(t.attempts))
    for t in run.tasks
])
print("events:")
for name in run.event_operations():
    print("   ", name)

## 8 — Result and evidence inspection

The Investigator's typed result carries provenance: an
`InvestigationReport` with evidence references (the read-before-cite
gate strips anything unread), the run's trace id, and measured local
usage. `NO_ANALYSIS` is the **expected** terminal here — with the
Analyst disabled the run stops instead of fabricating a finding.

In [ ]:
record = run.tasks[0]
report = record.output
print("task        :", record.task_id, record.status,
      "| trace:", record.agent_run_id)
print("investigation:", report.investigation_id,
      "| findings:", len(report.findings))
print("measured usage:", record.usage_total)
print("evidence sample:", report.findings[0].evidence_refs)
without_refs = [f for f in report.findings if not f.evidence_refs]
print("findings without evidence:", len(without_refs),
      "(inspected-evidence gate keeps provenance honest)")
print("files read  :", [path for path, _ in models["arch"].read_files])
print("router served:", router.labels)
print("merged report findings:", len(run.report.findings))
assert report.findings[0].evidence_refs, "provenance missing"
assert router.labels.get("P1") == "architecture"
assert run.finding is None, "no synthesized finding without an Analyst"

## 9 — Budget and call-count assertions

Every restriction is asserted in executable code here (the focused
tests re-run these cells and check the same facts). The last block
re-verifies that the H1 request-budget preflight still rejects an
oversized planner request **before** any client call, and reports
usage honestly: no provider usage exists for a dry run, and no
monetary cap is claimed.

In [ ]:
# 1) ZERO provider calls — enforced, not merely labeled
assert PROVIDER_GUARD.limit == 0
assert PROVIDER_GUARD.calls == 0
print("provider calls  : 0 (guard limit 0 — would have raised)")

# 2) exactly one task, one attempt, no retry, no replan
assert policy.executed == ["P1"] and policy.blocked == []
assert policy.execute_attempts == 1
assert len(run.tasks) == 1 and run.tasks[0].attempts == 1
assert run.retries == 0
ops = run.event_operations()
assert "orchestration.retry" not in ops
assert ops.count("orchestration.step.start") == 1
assert ops.count("orchestration.agent.start") == 1
assert [step.id for step in run.plan.steps] == ["P1"], "plan unchanged"
print("tasks/retry     : 1 executed, attempts=1, 0 retries, no replan")

# 3) Analyst and the second specialist never ran
assert analyst.calls == [] and run.finding is None
assert all(e["agent_role"] == "INVESTIGATOR" for e in run.agent_runs)
assert len(models["flow"].requests) == 0
print("analyst/flow    : never invoked (asserted)")

# 4) only offline double turns (no provider anywhere)
assert planner.model == "offline-scripted"
assert models["arch"].model == "offline-scripted"
print("offline turns   : planner={} investigator={} (local doubles)"
      .format(len(planner.requests), len(models["arch"].requests)))

# 5) H1 request-budget preflight still guards ANY planner path
class _BombClient:
    def __init__(self):
        self.calls = 0

    def complete(self, request):
        self.calls += 1
        raise AssertionError("over-budget request must never be sent")


bomb = _BombClient()
try:
    rq._build_plan(bomb, "x" * 400_000)   # incident-sized question
except RequestBudgetExceededError as exc:
    budget_message = str(exc)
else:
    raise SystemExit("expected RequestBudgetExceededError")
assert bomb.calls == 0
print("budget preflight : rejected locally, 0 client calls —",
      budget_message[:60])

# 6) usage honesty — nothing is invented
print("provider usage  : UNAVAILABLE (no provider call was made)")
print("measured local  :", record.usage_total)
print("cost            : NOT CALCULATED — no pricing source here;")
print("                  H1/H2 token estimates are NOT a monetary cap")

## 10 — Optional, explicitly enabled live-model experiment

**Disabled by default** (`LIVE_MODEL_TEST = False`). Credentials alone
never enable it: turning it on takes TWO deliberate edits (the flag
above AND `CONFIRM_LIVE_CALLS = True` in the cell below). Before any
paid call the cell displays provider/model, the estimated input size,
the experiment input ceiling, the output limit, and the remaining call
allowance.

The experiment keeps its limits SEPARATE from the application's
general request budget (H1/H2 — the global 95,232-token default is
unchanged and stays active as a second gate inside `_build_plan`):

- **output**: `max_tokens = 2048` — a realistic Planner allowance,
  stamped onto the request via the existing `LLMRequest` contract and
  translated by the provider adapter into the payload's own
  `max_tokens` parameter (verified by a focused test);
- **input**: an experiment-specific ESTIMATED ceiling of **8,000
  tokens**, measured with the SAME H1 estimator the application uses
  (`chars_per_token = 4` — a ratio, NOT a tokenizer, so the number is
  an ESTIMATE, never an exact token count). The ceiling is enforced
  TWICE locally: a probe measures the exact outgoing Planner request
  after construction and before any call, and `CallLimitedClient`
  re-enforces it at the client boundary. An over-ceiling request is
  REJECTED locally — never sent, never silently truncated;
- **calls**: `max_retries = 0` disables SDK-level retries and
  `CallLimitedClient(max_calls=1)` permits exactly ONE provider call
  (call #2 raises before the provider is touched). The blanket
  provider guard is relaxed from 0 to 1 only for the single call and
  re-armed to 0 in a `finally`;
- **roles**: the Planner is the one REAL model; the Investigator stays
  an offline fixture and the Analyst stays disabled, so graph routing
  can never invoke a second live model. The cell stops after the
  single Planner call — no graph execution, no second stage.

In [ ]:
import os

if not LIVE_MODEL_TEST:
    print("live experiment NOT executed (LIVE_MODEL_TEST = False).")
    print("The dry-run results above stand; the paid path stays off.")
elif not os.environ.get("OPENAI_API_KEY"):
    print("LIVE_MODEL_TEST is True but OPENAI_API_KEY is absent —")
    print("the live path stays disabled (dry run already recorded).")
else:
    from repo_intelligence.agent.request_budget import (
        CHARS_PER_TOKEN,
        RequestBudgetExceededError,
        RequestBudgetPolicy,
        enforce_request_budget_or_raise,
    )
    from repo_intelligence.llm.openai_client import (
        DEFAULT_MODEL,
        OpenAILLMClient,
    )
    from repo_intelligence.planning.planner import build_planning_request

    # experiment-specific ESTIMATED input ceiling (tokens). Separate
    # from the global H1 default (95,232): the SAME estimator +
    # enforcement machinery, a tighter policy object only — the
    # application's general request-budget policy is untouched.
    experiment_input_budget = RequestBudgetPolicy(
        max_input_tokens=LIVE_MAX_INPUT_TOKENS,
        context_capacity_tokens=None,   # explicit, not capacity-derived
    )

    # --- measure the EXACT outgoing Planner request BEFORE any call ----
    # build_planning_request is the same builder _build_plan uses, so
    # this probe is byte-identical to the request _build_plan would
    # send; the ceiling is established on the real request, not a guess.
    probe_request = build_planning_request(QUESTION)
    try:
        probe_request, probe_record = enforce_request_budget_or_raise(
            probe_request,
            experiment_input_budget,
            subject="live planner request",
        )
    except RequestBudgetExceededError as exc:
        probe_record = exc.record
        probe_rejected = True
    else:
        probe_rejected = False

    totals = probe_record["totals"]
    estimation = probe_record["estimation"]
    ceiling = probe_record["budget"]["effective_input_budget_tokens"]
    print("=== live configuration (displayed BEFORE any call) ===")
    print("provider        : OpenAI chat-completions")
    print("model           :", DEFAULT_MODEL)
    print("planner role    : REAL model (the one permitted call)")
    print("investigator    : offline fixture (0 provider calls)")
    print("analyst         : disabled")
    print("SDK retries     : 0 (client built with max_retries=0)")
    print("output limit    : max_tokens =", LIVE_MAX_OUTPUT_TOKENS)
    print("input ceiling   : ~{} ESTIMATED input tokens (experiment)"
          .format(ceiling))
    print("estimated input : ~{} tokens ({} chars / {} per token)"
          .format(totals["total_tokens_est"],
                  totals["total_characters"],
                  estimation["chars_per_token"]))
    print("estimate caveat : chars-per-token RATIO, not a tokenizer;"
          " provider-reported usage is shown AFTER the call")
    print("call allowance  : {} provider call(s) permitted (this call)"
          .format(LIVE_MAX_PROVIDER_CALLS))

    if probe_rejected:
        print("LOCAL REJECTION : oversized planner request — NOT sent")
        print("  reason        :",
              probe_record.get("rejection_reason"))
        print("  total_tokens_est={} vs ceiling={}".format(
            totals["total_tokens_est"], ceiling))
        print("  provider calls made: 0")
    elif not CONFIRM_LIVE_CALLS:
        print("CONFIRM_LIVE_CALLS is False — NO paid call made.")
        print("Set CONFIRM_LIVE_CALLS = True in this cell to call.")
    else:
        real_client = OpenAILLMClient(model=DEFAULT_MODEL, max_retries=0)
        live_planner = smoke_policy.CallLimitedClient(
            real_client,
            max_calls=LIVE_MAX_PROVIDER_CALLS,
            max_tokens=LIVE_MAX_OUTPUT_TOKENS,
            # the 8,000-token ceiling re-enforced on the exact
            # outgoing instance, at the client boundary (defense in
            # depth behind the pre-call probe above)
            input_budget=experiment_input_budget,
        )
        PROVIDER_GUARD.limit = LIVE_MAX_PROVIDER_CALLS   # exactly one
        try:
            live_plan = rq._build_plan(live_planner, QUESTION)  # call
        finally:
            PROVIDER_GUARD.limit = 0   # re-arm blanket dry-run protection
        usage = live_planner.responses[0].usage
        request0 = live_planner.requests[0]
        local_chars = sum(
            len(getattr(message, "content", "") or "")
            for message in request0.messages
        )
        print("live plan       :",
              [(step.id, step.goal[:40]) for step in live_plan.steps])
        print("provider usage  :", usage)
        print("local estimate  : ~{} tokens ({} chars / {})"
              .format(local_chars // CHARS_PER_TOKEN, local_chars,
                      CHARS_PER_TOKEN))
        print("cost            : UNAVAILABLE — no pricing source; not "
              "claimed")
        print("STOPPED after 1 call — no graph run, no second stage "
              "(by design)")

## 11 — Shared ReAct context-safety demonstration (offline)

The optional section-10 controls bound **one** Planner call. The
ORIGINAL failure was different: a ReAct-style loop kept calling the
model, every call appending tool observations to the history, until
ONE outgoing request exceeded the model's context limit. Planner-only
limits do not solve that general failure.

**This section demonstrates the shared, graph-agnostic controls** in
`repo_intelligence.react_safety` — usable by ANY ReAct agent or
graph, with no agent names, task names, or repository identifiers:

- `BudgetedLLMClient` — wraps any LLM client; the COMPLETE outgoing
  request (messages + tool definitions + response-format schema) is
  measured with the project's existing `agent.request_budget`
  machinery immediately before the provider boundary. Oversized →
  REJECTED locally (never sent, never silently truncated, never
  retried unchanged).
- `bound_observation` — deterministically bounds ONE tool observation
  before it joins history (truncation metadata + continuation hint).
  Per-observation bounds do NOT guarantee the total fits.
- `restrict_history` — per-request safety net: drops the OLDEST
  non-protected interactions from THAT request only (the transcript
  itself is never mutated), keeping tool call/result pairs whole.
- `ReActLoopGuard` — independent limits for iterations, tool calls,
  model calls, per-request input tokens, and a cumulative input-token
  allowance, with a clear terminal status on exhaustion.

**Integrated into the application path:** the same
`bound_observation` control now runs inside the production `Agent`
loop (`agent/agent.py`) — every tool observation is bounded BEFORE
it joins the conversation that the NEXT model request carries
(byte-identical for tool-protocol-compliant results). The loop's
existing H1/H2 preflight on every model call and its
`max_tool_calls` execution terminal remain the run's protections
— reused, not re-wrapped. Those seams are asserted on the real
loop by `tests/agent/test_react_integration.py`; the
demonstrations below exercise the identical classes offline.

Everything below runs OFFLINE with **fake model clients and fake
tools**: no credentials, no network, no dependence on
`OpenAILLMClient` (the armed class-level guard stays at limit 0 and
is never touched). These demonstrations are distinct from the
optional live Planner call in section 10 — there is no live
Investigator call anywhere in this notebook. Token figures are
ESTIMATES (chars-per-token ratio) unless a provider-reported usage
value is available; nothing here claims a monetary cap.

In [ ]:
# --- Section 11: shared ReAct context safety (OFFLINE, fake clients) ------
# Fake model clients + fake tools only: no credentials, no network, no
# OpenAILLMClient involved (the armed guard at limit 0 is untouched).
from repo_intelligence.react_safety import (
    BudgetedLLMClient,
    ReActLoopGuard,
    ReActSafetyConfig,
    ReActTermination,
    RequestBudgetExceededError,
    bound_observation,
    restrict_history,
)
from repo_intelligence.llm.models import (
    LLMRequest,
    LLMResponse,
    Message,
    StructuredOutputSpec,
    ToolCall,
    ToolDefinition,
    Usage,
)


class _FakeTool:
    def __init__(self, name, payload_fn):
        self.name = name
        self.payload_fn = payload_fn

    def run(self, arguments):
        return self.payload_fn(arguments)


class _FakeClient:
    """Scripted responses: *turns* tool-call turns, then an answer."""

    def __init__(self, turns, answer="done"):
        self._turns = turns
        self._answer = answer
        self.seen = []          # requests that reached the provider
        self.model = "offline-fake-model"

    def complete(self, request):
        self.seen.append(request)
        index = len(self.seen)
        if index <= self._turns:
            message = Message(
                role="assistant",
                content="",
                tool_calls=[ToolCall(
                    id="c{}".format(index),
                    name="fake_tool",
                    arguments={"i": index},
                )],
            )
            return LLMResponse(
                message=message,
                finish_reason="tool_calls",
                usage=Usage(input_tokens=7, output_tokens=3,
                            total_tokens=10),
            )
        return LLMResponse(
            message=Message(role="assistant", content=self._answer),
            finish_reason="stop",
            usage=Usage(input_tokens=7, output_tokens=3, total_tokens=10),
        )


def _mini_react(client, tool, question, config, payload_fn):
    """A tiny generic ReAct loop using ONLY the shared controls.

    No agent-name checks, no task-name checks: the SAME three shared
    objects (budget client, loop guard, observation bounder) protect
    this loop exactly as they would any other agent or graph.
    """
    budgeted = BudgetedLLMClient(
        client, policy=config.request_budget_policy()
    )
    guard = ReActLoopGuard(config)
    messages = [
        Message(role="system", content="react contract (protected)"),
        Message(role="user", content=question),
    ]
    history_growth = []
    rejections = 0
    while True:
        if not guard.allow_iteration():
            break
        guard.begin_iteration()
        try:
            response = budgeted.complete(
                LLMRequest(messages=list(messages))
            )
        except RequestBudgetExceededError:
            guard.record_budget_rejection(budgeted.rejected[-1])
            rejections += 1
            break
        guard.record_model_response(
            response, budget_record=budgeted.records[-1]
        )
        messages.append(response.message)
        history_growth.append(sum(
            len(m.content or "") for m in messages
        ))
        if response.finish_reason != "tool_calls":
            break
        for call in response.message.tool_calls:
            if not guard.allow_tool_call():
                break
            bounded, _obs = bound_observation(
                payload_fn(call),
                config.max_observation_chars,
                label=tool.name,
            )
            guard.record_tool_call()
            messages.append(Message(role="tool", content=bounded))
    guard.stop(guard.termination_reason or ReActTermination.COMPLETED)
    return budgeted, guard, messages, history_growth, rejections


# -- demo 1 + 2: consecutive iterations, observations grow -------------------
cfg = ReActSafetyConfig(
    max_iterations=6,
    max_tool_calls=6,
    max_model_calls=8,
    max_observation_chars=4000,
    max_request_input_tokens=12000,
)
tool_a = _FakeTool(
    "fake_tool",
    lambda a: ("evidence row {}\n".format(a["i"])) * 120,
)
client_a = _FakeClient(turns=3, answer="first agent done")
budgeted_a, guard_a, msgs_a, growth_a, rejects_a = _mini_react(
    client_a, tool_a, "investigate components", cfg,
    lambda call: tool_a.run(call.arguments),
)
print("D1-2 iterations:", guard_a.iterations,
      "| model calls:", guard_a.model_calls,
      "| tool calls:", guard_a.tool_calls)
print("    history chars growth:", growth_a)
print("    terminal status:", guard_a.termination_reason,
      "| provider calls:", len(client_a.seen))
assert guard_a.iterations >= 3, "must show consecutive iterations"
assert growth_a == sorted(growth_a) and growth_a[-1] > growth_a[0], \
    "observations must grow across iterations"
assert guard_a.termination_reason == ReActTermination.COMPLETED
assert rejects_a == 0

# -- demo 3: a single oversized observation is bounded ------------------------
huge = "X" * 50_000
bounded, obs_record = bound_observation(huge, 4000, label="big_tool")
print("D3 observation bound: {} -> {} chars (limit {}, truncated={})"
      .format(obs_record["original_chars"],
              obs_record["returned_chars"],
              obs_record["char_limit"], obs_record["truncated"]))
assert obs_record["truncated"] is True
assert len(bounded) <= 4000, "bounded observation must fit its ceiling"
assert obs_record["note"], "truncation must leave a continuation hint"

# -- demo 4: individually-fitting observations still get budget-enforced -----
grow_cfg = ReActSafetyConfig(
    max_iterations=8,
    max_tool_calls=8,
    max_model_calls=8,
    max_observation_chars=2000,        # each observation fits THIS cap
    max_request_input_tokens=1000,     # 4,000 chars total request
)
client_b = _FakeClient(turns=4, answer="grown run done")
budgeted_b, guard_b, msgs_b, growth_b, rejects_b = _mini_react(
    client_b, _FakeTool("t", lambda a: (
        "evidence {}\n".format(a["i"])) + "row\n" * 900),
    "grow", grow_cfg,
    lambda call: ("evidence {}\n".format(call.arguments["i"]))
    + "row\n" * 900,
)
dispositions = [r["disposition"] for r in budgeted_b.records]
sent_records = [
    r for r in budgeted_b.records
    if r["disposition"] in ("accepted", "reduced")
]
budget_chars = grow_cfg.request_budget_policy() \
    .effective_input_budget_characters
history_chars = sum(len(m.content or "") for m in msgs_b)
print("D4 per-observation cap: {} chars | request budget: {} chars"
      .format(grow_cfg.max_observation_chars, budget_chars))
print("    dispositions:", dispositions)
print("    history chars:", history_chars,
      "-> combined history EXCEEDS the budget:",
      history_chars > budget_chars)
assert "reduced" in dispositions or "rejected" in dispositions, \
    "the boundary must intervene once the combined history overflows"
assert sent_records and all(
    r["within_budget"] for r in sent_records
), "every request allowed to be sent must fit the budget"
assert history_chars > budget_chars, \
    "individual caps alone did NOT keep the total under budget"

# -- demo 5: oversized COMPLETE request rejected before the provider ----------
oversized_cfg = ReActSafetyConfig(max_request_input_tokens=1000)
client_c = _FakeClient(turns=1)
budgeted_c = BudgetedLLMClient(
    client_c, policy=oversized_cfg.request_budget_policy(),
    subject="react request",
)
oversized = LLMRequest(
    messages=[
        Message(role="system", content="S" * 9000),
        Message(role="user", content="q"),
    ],
    tools=[ToolDefinition(
        name="big_tool",
        description="D" * 6000,
        input_schema={"type": "object"},
    )],
    structured_output=StructuredOutputSpec(
        name="Result", schema={"type": "object", "properties": {}}
    ),
)
try:
    budgeted_c.complete(oversized)
except RequestBudgetExceededError as exc:
    rejection = exc.record
else:
    raise AssertionError("oversized complete request must be rejected")
print("D5 rejected BEFORE provider: reason={}, est={} tokens vs "
      "ceiling={} tokens".format(
          rejection.get("rejection_reason"),
          rejection["totals"]["total_tokens_est"],
          rejection["budget"]["effective_input_budget_tokens"]))
print("    provider calls:", budgeted_c.calls_reached_provider,
      "| sent:", len(budgeted_c.sent))
assert budgeted_c.calls_reached_provider == 0
assert budgeted_c.sent == []
assert rejection["totals"]["total_tokens_est"] > \
    rejection["budget"]["effective_input_budget_tokens"]

# -- demo 6: per-request history restriction preserves provenance -------------
transcript = [
    Message(role="system", content="SYS contract + provenance rules"),
    Message(role="user", content="the question"),
]
for step in range(6):
    transcript.append(Message(
        role="assistant", content="",
        tool_calls=[ToolCall(id="s{}".format(step), name="t",
                             arguments={"i": step})],
    ))
    transcript.append(Message(
        role="tool", content="obs {} {}".format(step, "y" * 4000)
    ))
    transcript.append(Message(
        role="user", content="proceed to step {}".format(step)
    ))
original_len = len(transcript)
kept, record = restrict_history(transcript, max_chars=8000,
                                keep_recent=1)
pairs_whole = (
    sum(1 for m in kept if m.role == "tool")
    == sum(1 for m in kept
           if m.role == "assistant" and m.tool_calls)
)
print("D6 restrict_history: dropped {dropped_units} units / "
      "{dropped_chars} chars; system kept: {sys}; pairs whole: "
      "{pairs}; within limit: {within_limit}".format(
          sys=any(m.role == "system" for m in kept),
          pairs=pairs_whole, **record))
assert len(transcript) == original_len, "transcript must not mutate"
assert any(m.role == "system" for m in kept), "protected kept"
assert pairs_whole, "tool call/result pairs kept whole"
assert record["within_limit"] is True
assert record["dropped_units"] >= 1

# -- demo 7: iteration / tool-call exhaustion -> clear terminal status --------
iter_guard = ReActLoopGuard(ReActSafetyConfig(max_iterations=1))
iter_guard.allow_iteration()
iter_guard.begin_iteration()
assert not iter_guard.allow_iteration()
iter_guard.stop(iter_guard.termination_reason
                or ReActTermination.ITERATION_LIMIT_REACHED)
print("D7 iteration exhaustion ->", iter_guard.terminal_status())

tool_guard = ReActLoopGuard(ReActSafetyConfig(max_tool_calls=1))
tool_guard.allow_tool_call()
tool_guard.record_tool_call()
assert not tool_guard.allow_tool_call()
assert not tool_guard.allow_iteration(), \
    "a terminal status stops every later action"
print("D7 tool-call exhaustion ->", tool_guard.terminal_status())
assert tool_guard.termination_reason == \
    ReActTermination.TOOL_CALL_LIMIT_REACHED

# -- demo 8: a SECOND dummy agent uses the SAME shared controls ---------------
client_d = _FakeClient(turns=2, answer="second agent done")
tool_d = _FakeTool("different_tool", lambda a: "other evidence line")
budgeted_d, guard_d, msgs_d, growth_d, rejects_d = _mini_react(
    client_d, tool_d, "another task entirely", cfg,
    lambda call: tool_d.run(call.arguments),
)
print("D8 second dummy agent:", guard_d.iterations, "iterations,",
      guard_d.model_calls, "model calls, status",
      guard_d.termination_reason)
assert isinstance(budgeted_d, BudgetedLLMClient) \
    and isinstance(guard_d, ReActLoopGuard)
assert type(budgeted_d) is type(budgeted_a) \
    and type(guard_d) is type(guard_a), \
    "identical shared classes — no per-agent variants"
assert guard_d.termination_reason == ReActTermination.COMPLETED
print("shared policy (same classes for both):",
      type(cfg.request_budget_policy()).__name__)
print("section 11 complete — all demos OFFLINE, provider calls: 0 "
      "(fake clients only; the armed guard was never touched)")
assert PROVIDER_GUARD.calls == 0, "section 11 must make no provider call"

## 12 — Results, limitations, and next steps

### What this notebook proves (offline, deterministic)

- Planner → **one** valid fixture plan (Plan 1), through the H1-gated
  `_build_plan` seam.
- Selection → `ready_tasks` returns exactly `P1`; the router classifies
  P1 to the architecture specialist; roles preassigned as INVESTIGATOR.
- Execution → ONE graph run, one task record (`attempts=1`), typed
  `InvestigationReport` (≈151 findings, evidence refs on findings,
  trace id, measured local usage: tool calls + model iterations).
- Termination → `INCOMPLETE / NO_ANALYSIS` immediately after the
  Investigator result: **expected**, because the Analyst is disabled —
  no synthesized finding is fabricated.
- Restrictions asserted in code: provider calls 0, retries 0, no
  replan, Analyst never invoked, second specialist never touched,
  one-task policy recorded `executed == ["P1"]`.
- H1 preflight re-proven live in section 9: a 400k-char planner
  request is rejected locally with **0** client calls.
- Section 11 proves the GENERAL failure mode is closed by the shared
  `react_safety` controls — not by Planner-only limits:
  consecutive ReAct iterations with growing observations, a single
  oversized observation bounded, combined history enforced at the
  request boundary, an oversized complete request rejected BEFORE the
  provider, provenance-preserving per-request restriction, iteration/
  tool exhaustion with a clear terminal status, and a SECOND dummy
  agent protected by the identical shared classes.

- The production-path integration is proven OUTSIDE this notebook
  by `tests/agent/test_react_integration.py` (observation bounded
  on the real `Agent` loop, independent per-request budget
  records, byte-identical pass-through for compliant results,
  execution-limit terminal intact) and by the graph workflow's
  per-iteration budget records
  (`test_repository_query_langgraph.py::test_safety_records_
  ride_every_model_call_of_the_graph_workflow`).

### Limitations (honest)

- **The optional section-10 live Planner call does NOT validate
  the multi-agent workflow**: it exercises ONE planner request
  under a real provider (when explicitly enabled). Workflow
  correctness is what the offline graph execution above proves —
  Planner plan, dispatched Investigator work, Analyst handoffs,
  and terminal state — plus the offline integration tests. No
  live Investigator or Analyst call exists anywhere in this
  notebook.
- **No live-provider verification in this environment** (no
  `OPENAI_API_KEY`): section 10's paid path is implemented and guarded
  but was not exercised here; its only unverified aspects are the real
  model's plan quality and its reported usage numbers.
- The graph itself has no “stop after N tasks” switch — the max-1 rule
  lives in the test-only `OneTaskExecutionPolicy` (documented above);
  without it, a multi-step plan would continue past task 1 (also
  asserted by a focused test that feeds a 2-step plan to the policy).
- `NO_ANALYSIS` means the restricted flow stopped honestly — running
  the Analyst requires a later, explicitly enabled experiment.
- The experiment input ceiling (8,000 tokens) and every token figure
  in section 11 are **ESTIMATES** from a chars-per-token RATIO (4 chars
  ≈ 1 token), not a tokenizer and not a provider-side count — they
  bound requests conservatively but do not guarantee exact token
  usage; provider-reported usage is displayed after a real call for
  comparison.
- The ceiling is enforced on the EXACT outgoing request (probe after
  construction + re-enforcement at the client boundary), so an
  oversized request is rejected locally BEFORE the provider is
  invoked — nothing is silently truncated.
- **Per-request limits are NOT cumulative run limits**: a request
  ceiling bounds EACH request; `ReActLoopGuard`'s iteration, tool,
  model-call, and cumulative-token limits bound the RUN. Both are
  demonstrated in section 11.
- **No monetary cap is claimed.** H1/H2 (global) and the experiment
  ceiling enforce TOKEN budgets; dollar cost depends on provider
  pricing which nothing in this repository enforces.
  `max_tokens=2048` bounds output length, `max_retries=0` bounds SDK
  retries, `CallLimitedClient` bounds call count — that is the full
  set of live cost controls.

### Next step

Enable exactly one real model call under the section-10 controls,
inspect the provider-reported usage against the local estimate, and
only THEN consider a second call or a second agent — never both at
once.